# Multimodal SFT: Chart Reasoning

Train `gpt-4.1-2025-04-14` on the exact 50-row training and 20-row validation
files from the current-branch ChartQA source demo. This workflow validates the
source bytes and submits or resumes one
`GlobalStandard` supervised fine-tuning job, monitors it to a terminal state,
and downloads service-emitted metrics.

It does not deploy or invoke the fine-tuned model and performs no held-out
inference evaluation.

Actual results may vary by model version, data, configuration, region availability, and service conditions.

## 1. Configure the training run

Copy `../.env.template` to `../.env`, set the Foundry project endpoint, run
`az login` before executing the client and training cells, which can create a
paid job. Set `FOUNDRY_RUN_ID` to resume an interrupted run without duplicate
uploads or submission. The `.env` is local to this module, not the repository root. Job creation disables SDK retries.

In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import os
import re
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

PROJECT_ENDPOINT = os.environ.get("FOUNDRY_PROJECT_ENDPOINT", "").strip()
MODEL = "gpt-4.1-2025-04-14"
TRAINING_TYPE = "GlobalStandard"
TERMINAL_STATES = {"succeeded", "failed", "cancelled"}

default_run_id = f"chartqa-{datetime.now(timezone.utc):%Y%m%d-%H%M%S}-{uuid.uuid4().hex[:8]}"
RUN_ID = os.environ.get("FOUNDRY_RUN_ID", "").strip() or default_run_id
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._-]{0,63}", RUN_ID):
    raise ValueError("FOUNDRY_RUN_ID must use 1-64 letters, numbers, dots, underscores, or hyphens.")

RUN_DIR = ROOT / "outputs" / "runs" / RUN_ID
STATE_PATH = RUN_DIR / "run-state.json"
RUN_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = {
    "train": {
        "path": ROOT / "data" / "preserved" / "chart-qa-run-20261002-v4-train.jsonl",
        "rows": 50,
        "bytes": 4_380_971,
        "sha256": "2024e5d28f1d7b4c8027b6842196c3f5bcaf105f250bf8bc86a3520b9aa8c5f8",
    },
    "validation": {
        "path": ROOT / "data" / "preserved" / "chart-qa-run-20261002-v4-val.jsonl",
        "rows": 20,
        "bytes": 1_852_980,
        "sha256": "6b569a58c36f4955a33a4cbb79a90a40e63a2f399ed13bf9877dc62eda74e58b",
    },
}

print(f"Run ID: {RUN_ID}")
print(f"Model: {MODEL}")
print(f"Training type: {TRAINING_TYPE}")
print("Run the upload/training cell to submit or resume a paid job.")

## 2. Verify the exact source-demo train and validation files

In [ ]:
def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def validate_jsonl(label: str, spec: dict) -> dict:
    path = spec["path"]
    actual = {
        "split": label,
        "path": str(path.relative_to(ROOT)),
        "rows": 0,
        "bytes": path.stat().st_size,
        "sha256": sha256(path),
    }
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            record = json.loads(line)
            messages = record.get("messages", [])
            if [message.get("role") for message in messages] != ["system", "user", "assistant"]:
                raise ValueError(f"{label} row {line_number} has an unexpected message sequence.")
            content = messages[1].get("content", [])
            image_urls = [
                item.get("image_url", {}).get("url", "")
                for item in content
                if item.get("type") == "image_url"
            ]
            if len(image_urls) != 1 or not image_urls[0].startswith("data:image/jpeg;base64,"):
                raise ValueError(f"{label} row {line_number} does not contain one JPEG data URL.")
            actual["rows"] += 1

    for field in ("rows", "bytes", "sha256"):
        if actual[field] != spec[field]:
            raise ValueError(
                f"{label} {field} mismatch: {actual[field]!r} != {spec[field]!r}"
            )
    return actual


dataset_evidence = [validate_jsonl(label, spec) for label, spec in DATASETS.items()]
display(pd.DataFrame(dataset_evidence))

## 3. Verify the compatible Foundry project

In [ ]:
credential = None
project_client = None
client = None
if not PROJECT_ENDPOINT:
    raise RuntimeError("Set FOUNDRY_PROJECT_ENDPOINT in .env before running training.")
credential = DefaultAzureCredential(process_timeout=60)
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
client = project_client.get_openai_client(default_headers={"Accept-Encoding": "identity"})


## 4. Submit once and monitor to a terminal state

In [ ]:
def load_state() -> dict:
    if not STATE_PATH.exists() or STATE_PATH.stat().st_size == 0:
        return {}
    return json.loads(STATE_PATH.read_text(encoding="utf-8"))


def save_state(state: dict) -> None:
    state["updated_at"] = datetime.now(timezone.utc).isoformat()
    temporary = STATE_PATH.with_suffix(".tmp")
    temporary.write_text(json.dumps(state, indent=2, sort_keys=True), encoding="utf-8")
    temporary.replace(STATE_PATH)


state = load_state()
if state and state.get("run_id") != RUN_ID:
    raise RuntimeError(f"Run-state ID mismatch in {STATE_PATH}.")
if state:
    expected = {
        "project_endpoint": PROJECT_ENDPOINT,
        "model": MODEL,
        "training_type": TRAINING_TYPE,
        "datasets": dataset_evidence,
    }
    for key, value in expected.items():
        if state.get(key) != value:
            raise RuntimeError(f"Run-state mismatch for {key}; choose a new FOUNDRY_RUN_ID.")
if not state:
    state = {
        "run_id": RUN_ID,
        "created_at": datetime.now(timezone.utc).isoformat(),
        "project_endpoint": PROJECT_ENDPOINT,
        "model": MODEL,
        "training_type": TRAINING_TYPE,
        "datasets": dataset_evidence,
    }
    save_state(state)


def wait_for_file(file_id: str, label: str):
    uploaded = client.files.retrieve(file_id)
    started = time.monotonic()
    while uploaded.status not in {"processed", "error", "failed", "cancelled", "expired", "deleted"}:
        if time.monotonic() - started >= 900:
            raise TimeoutError(f"Timed out processing {label} file {file_id}.")
        print(f"{label} file {file_id}: {uploaded.status}", flush=True)
        time.sleep(15)
        uploaded = client.files.retrieve(file_id)
    print(f"{label} file {file_id}: {uploaded.status}", flush=True)
    if uploaded.status != "processed":
        raise RuntimeError(f"{label} file import failed: {uploaded.status_details}")
    return uploaded


def ensure_uploaded(label: str, path: Path) -> str:
    key = f"{label}_file_id"
    if state.get(key):
        return wait_for_file(state[key], label).id

    created_after = int(datetime.fromisoformat(state["created_at"]).timestamp()) - 5
    matching_files = [
        candidate
        for candidate in client.files.list(limit=100, order="desc")
        if (
            candidate.filename == path.name
            and candidate.bytes == path.stat().st_size
            and candidate.created_at >= created_after
        )
    ]
    if len(matching_files) > 1:
        raise RuntimeError(f"Found multiple matching {label} uploads for run ID {RUN_ID}.")
    if matching_files:
        uploaded = matching_files[0]
    else:
        with path.open("rb") as handle:
            uploaded = client.files.create(file=handle, purpose="fine-tune")

    state[key] = uploaded.id
    save_state(state)
    return wait_for_file(uploaded.id, label).id


ft_job = None
training_file_id = ensure_uploaded("training", DATASETS["train"]["path"])
validation_file_id = ensure_uploaded("validation", DATASETS["validation"]["path"])

if state.get("job_id"):
    ft_job = client.fine_tuning.jobs.retrieve(state["job_id"])
else:
    submission_was_pending = state.get("submission_pending", False)
    state["submission_pending"] = True
    save_state(state)
    matching_jobs = []
    for candidate in client.fine_tuning.jobs.list(limit=100):
        if (
            candidate.suffix == RUN_ID
            and candidate.model == MODEL
            and candidate.training_file == training_file_id
            and candidate.validation_file == validation_file_id
        ):
            candidate_method = getattr(candidate, "method", None)
            if hasattr(candidate_method, "model_dump"):
                candidate_method = candidate_method.model_dump()
            if not isinstance(candidate_method, dict):
                raise RuntimeError("Matching job does not expose its training recipe; reconcile it before reuse.")
            supervised = candidate_method.get("supervised") or {}
            candidate_epochs = (supervised.get("hyperparameters") or {}).get("n_epochs")
            if candidate_method.get("type") != "supervised" or getattr(candidate, "seed", None) != 42 or candidate_epochs != 1:
                raise RuntimeError("Matching job has a different seed or epoch recipe; refusing reuse or duplicate creation.")
            matching_jobs.append(candidate)
    if len(matching_jobs) > 1:
        raise RuntimeError(f"Found multiple matching jobs for run ID {RUN_ID}.")
    if matching_jobs:
        ft_job = matching_jobs[0]
    else:
        if submission_was_pending:
            raise RuntimeError("Previous submission outcome is unknown; reconcile the remote job before retrying or clearing pending state.")
        ft_job = client.with_options(max_retries=0).fine_tuning.jobs.create(
            suffix=RUN_ID,
            training_file=training_file_id,
            validation_file=validation_file_id,
            model=MODEL,
            seed=42,
            method={
                "type": "supervised",
                "supervised": {
                    "hyperparameters": {
                        "n_epochs": 1,
                        "batch_size": None,
                        "learning_rate_multiplier": None,
                    }
                },
            },
            extra_body={"trainingType": TRAINING_TYPE},
        )
    state["job_id"] = ft_job.id
    state["submission_pending"] = False
    save_state(state)

print(f"{ft_job.id}: {ft_job.status}", flush=True)
started = time.monotonic()
while ft_job.status not in TERMINAL_STATES:
    if time.monotonic() - started >= 24 * 60 * 60:
        raise TimeoutError("Training monitoring timed out; resume with the same FOUNDRY_RUN_ID.")
    time.sleep(60)
    ft_job = client.fine_tuning.jobs.retrieve(ft_job.id)
    print(f"{ft_job.id}: {ft_job.status}", flush=True)

job_data = ft_job.to_dict()
state["status"] = ft_job.status
state["job"] = job_data
save_state(state)
print(f"{ft_job.id}: terminal status {ft_job.status}", flush=True)


## 5. Download service training metrics

In [ ]:
def last_non_null(frame: pd.DataFrame, column: str):
    if column not in frame:
        return None
    values = frame[column].dropna()
    return None if values.empty else float(values.iloc[-1])


if ft_job is not None:
    if ft_job.status != "succeeded":
        error = ft_job.error.to_dict() if ft_job.error else None
        raise RuntimeError(f"Fine-tuning ended with {ft_job.status}: {error}")

    result_files = ft_job.to_dict().get("result_files", [])
    if not result_files:
        raise RuntimeError("Succeeded job did not expose a result file.")

    downloaded = []
    for index, file_id in enumerate(result_files):
        output_path = RUN_DIR / f"result-{index}-{file_id}.csv"
        output_path.write_bytes(client.files.content(file_id).content)
        downloaded.append({"file_id": file_id, "path": str(output_path.relative_to(ROOT))})

    metrics = pd.read_csv(RUN_DIR / f"result-0-{result_files[0]}.csv")
    metric_columns = [
        column
        for column in (
            "step",
            "train_loss",
            "train_mean_token_accuracy",
            "valid_loss",
            "valid_mean_token_accuracy",
            "full_valid_loss",
            "full_valid_mean_token_accuracy",
            "timestamp",
        )
        if column in metrics.columns
    ]
    display(metrics[metric_columns].tail())

    summary = {
        "job_id": ft_job.id,
        "status": ft_job.status,
        "model": ft_job.model,
        "fine_tuned_model": ft_job.fine_tuned_model,
        "trained_tokens": ft_job.trained_tokens,
        "result_files": downloaded,
        "metrics": {
            "final_train_loss": last_non_null(metrics, "train_loss"),
            "final_train_mean_token_accuracy": last_non_null(
                metrics, "train_mean_token_accuracy"
            ),
            "final_validation_loss": last_non_null(metrics, "full_valid_loss"),
            "final_validation_mean_token_accuracy": last_non_null(
                metrics, "full_valid_mean_token_accuracy"
            ),
        },
    }
    (RUN_DIR / "metrics-summary.json").write_text(
        json.dumps(summary, indent=2, sort_keys=True), encoding="utf-8"
    )
    state["metrics_summary"] = summary
    save_state(state)
    display(pd.DataFrame([summary["metrics"]]))

if client is not None:
    client.close()
if project_client is not None:
    project_client.close()
if credential is not None:
    credential.close()